# nb_semantic_model_ga4

Import this file into Fabric as **nb_semantic_model_ga4**.

Creates the Direct Lake semantic model **GA4 Ecommerce** on lakehouse **GA4Lakehouse** (workspace **GA4-Report**). Writes two small conformed dimensions (`dbo.dim_ga4_channel`, `dbo.dim_ga4_device`), rebuilds `dbo.dim_ga4_page_type` only when a fact key is missing, then generates the model, adds relationships, marks `Date` as the date table, creates the calculated **`_Measures`** table, and upserts the base measures. No report is created. No secrets and no hardcoded workspace or lakehouse IDs.

**Kernel:** **Python 3.12** (not PySpark). Semantic Link / `sempy_labs` do not need Spark, and a Spark session can hit trial HTTP 430.

**Before Run all**

1. Attach default lakehouse **GA4Lakehouse** (Add data items → From OneLake catalog).
2. Set the kernel to **Python 3.12**.
3. Capacity must allow **XMLA read/write** (needed for `connect_semantic_model(..., readonly=False)`). If that call fails, enable XMLA Read/Write on the trial capacity and rerun from the TOM cell — the generate step may already have created the model.
4. Run this notebook only after the gold notebooks, including **`nb_gold_ga4_dim_date`**. Required tables: `dbo.dim_ga4_date`, `dbo.dim_ga4_page_type`, `dbo.gold_ga4_sessions`, `dbo.gold_ga4_daily_kpis`, `dbo.gold_ga4_funnel_daily`, `dbo.gold_ga4_items_daily`, `dbo.gold_ga4_landing_pages`, `dbo.gold_ga4_page_sessions`. `dbo.bronze_ga4_events` is left out of the model.

If a model named exactly `GA4 Ecommerce` already exists, this notebook **does not delete or overwrite it**. It skips `generate_direct_lake_semantic_model` and updates relationships and measures in place via TOM.


In [ ]:
%pip install -q semantic-link-labs


In [ ]:
# Parameters
WORKSPACE = "GA4-Report"
LAKEHOUSE = "GA4Lakehouse"
DATASET = "GA4 Ecommerce"

EXPECTED = {
    "Sessions": 360_129,
    "Revenue": 362_165,
    "Pageviews": 1_350_428,
}

# Friendly model table name → schema.table in the lakehouse.
# bronze_ga4_events is intentionally omitted.
TABLES = {
    "Date": "dbo.dim_ga4_date",
    "Channel": "dbo.dim_ga4_channel",
    "Device": "dbo.dim_ga4_device",
    "Page Type": "dbo.dim_ga4_page_type",
    "Sessions": "dbo.gold_ga4_sessions",
    "Daily KPIs": "dbo.gold_ga4_daily_kpis",
    "Funnel Daily": "dbo.gold_ga4_funnel_daily",
    "Items Daily": "dbo.gold_ga4_items_daily",
    "Landing Pages": "dbo.gold_ga4_landing_pages",
    "Page Sessions": "dbo.gold_ga4_page_sessions",
}

# Many → one, single-direction. One active path per dim-to-fact pair.
RELATIONSHIPS = [
    ("Sessions", "session_date", "Date", "date"),
    ("Daily KPIs", "session_date", "Date", "date"),
    ("Funnel Daily", "session_date", "Date", "date"),
    ("Items Daily", "event_date", "Date", "date"),
    ("Landing Pages", "session_date", "Date", "date"),
    ("Page Sessions", "event_date", "Date", "date"),
    ("Sessions", "channel_group", "Channel", "channel_group"),
    ("Daily KPIs", "channel_group", "Channel", "channel_group"),
    ("Funnel Daily", "channel_group", "Channel", "channel_group"),
    ("Sessions", "device_category", "Device", "device_category"),
    ("Daily KPIs", "device_category", "Device", "device_category"),
    ("Funnel Daily", "device_category", "Device", "device_category"),
    ("Sessions", "landing_page_path", "Page Type", "page_path"),
    ("Landing Pages", "landing_page_path", "Page Type", "page_path"),
    ("Page Sessions", "page_path", "Page Type", "page_path"),
]

# Lakehouse tables that must exist before the model is generated.
REQUIRED_TABLES = {
    "dim_ga4_date": "Run nb_gold_ga4_dim_date first.",
    "dim_ga4_page_type": "Run nb_gold_ga4_marketing first.",
    "gold_ga4_sessions": "Run nb_gold_ga4_marketing first.",
    "gold_ga4_daily_kpis": "Run nb_gold_ga4_marketing first.",
    "gold_ga4_funnel_daily": "Run nb_gold_ga4_marketing first.",
    "gold_ga4_items_daily": "Run nb_gold_ga4_marketing first.",
    "gold_ga4_landing_pages": "Run nb_gold_ga4_marketing first.",
    "gold_ga4_page_sessions": "Run nb_gold_ga4_page_sessions first.",
}


In [ ]:
from __future__ import annotations

import sempy.fabric as fabric
from sempy_labs.directlake import generate_direct_lake_semantic_model
from sempy_labs.tom import connect_semantic_model
from sempy_labs import refresh_semantic_model
from sempy_labs.lakehouse import lakehouse_attached

print("workspace:", WORKSPACE)
print("lakehouse attached:", lakehouse_attached())
if not lakehouse_attached():
    print(
        "WARN: attach GA4Lakehouse as the default lakehouse, then rerun. "
        "The generate call still passes source/workspace explicitly."
    )


## 0 · Conformed dimensions

Writes `dbo.dim_ga4_channel` and `dbo.dim_ga4_device` (distinct values unioned from the session-grained facts). Checks that every `page_path` / `landing_page_path` in the page facts is in `dbo.dim_ga4_page_type`. If any key is missing, rebuilds that dim as the union of paths using the same page-type rules as `nb_gold_ga4_marketing`, with `'other'` as the fallback.


In [ ]:
# Write dbo.dim_ga4_channel and dbo.dim_ga4_device. Rebuild dim_ga4_page_type if needed.
import os
import subprocess
import sys
from pathlib import Path

import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

ROOT = Path(os.environ.get("LAKEHOUSE_DEFAULT", "/lakehouse/default"))

# Same path and page-type rules as nb_gold_ga4_marketing.
PAGE_PATH_SQL = """
COALESCE(
    NULLIF(
        regexp_replace(
            regexp_replace({col}, '[?#].*$', ''),
            '^[a-zA-Z][a-zA-Z0-9+.-]*://[^/]+',
            ''
        ),
        ''
    ),
    '/'
)"""

CATEGORY_SLUGS = """
'apparel', 'mens', 'womens', 'hats', 'headgear', 'lifestyle', 'drinkware',
'bags', 'small goods', 'clearance', 'new', 'campus collection', 'eco friendly',
'shop by brand', 'youtube', 'google', 'accessories', 'fun', 'stationery',
'office', 'wearables', 'electronics', 'brands', 'notebooks journals',
'writing', 'stickers', 'gift cards', 'kids', 'nest', 'nest usa', 'spring sale',
'sale', 'android', 'google redesign', 'google rmg redesign', 'mens t shirts',
'men s t shirts', 'womens t shirts', 'women s t shirts', 'i am remarkable'
"""
PAGE_TYPE_SQL = f"""
CASE
    WHEN path_norm IS NULL OR path_norm IN ('', '/') OR path_norm IN ('/home', '/home 2', '/home.')
        THEN 'home'
    WHEN path_norm LIKE '%basket%'
      OR path_norm LIKE '%/cart'
      OR path_norm LIKE '%/cart/%'
      OR path_norm LIKE '%shop.axd/cart%'
        THEN 'cart'
    WHEN path_norm LIKE '%yourinfo%'
      OR path_norm LIKE '%payment.html%'
      OR path_norm LIKE '%revieworder%'
      OR path_norm LIKE '%ordercompleted%'
      OR path_norm LIKE '%checkout%'
        THEN 'checkout'
    WHEN path_norm LIKE '%asearch%'
      OR path_norm LIKE '%search%'
        THEN 'search'
    WHEN path_norm LIKE '%signin%'
      OR path_norm LIKE '%myaccount%'
      OR path_norm LIKE '%register%'
      OR path_norm LIKE '%forgot_password%'
      OR path_norm LIKE '%resetpassword%'
      OR path_norm LIKE '%wishlist%'
        THEN 'account'
    WHEN path_norm LIKE '%.axd%'
        THEN 'product'
    WHEN (
        path_norm LIKE '%google%sign%'
        OR path_norm LIKE '/eco/%'
        OR path_norm LIKE '/eco'
    ) AND last_seg NOT IN ({CATEGORY_SLUGS})
        THEN 'product'
    WHEN path_norm LIKE '%google%sign%'
      OR path_norm LIKE '%/store.html%'
      OR path_norm LIKE '/new/%'
      OR path_norm LIKE '/clearance/%'
      OR path_norm LIKE '/eco/%'
      OR path_norm LIKE '/eco'
        THEN 'category'
    ELSE 'other'
END
"""


def _storage():
    """abfss + bearer token in Fabric; local filesystem path otherwise."""
    try:
        import notebookutils
        ctx = notebookutils.runtime.context
        ws = ctx["defaultLakehouseWorkspaceId"]
        lh = ctx["defaultLakehouseId"]
        endpoint = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
        base = f"abfss://{ws}@{endpoint}/{lh}/Tables/dbo"
        opts = {
            "bearer_token": notebookutils.credentials.getToken("storage"),
            "use_fabric_endpoint": "true",
        }
        return base, opts
    except Exception:
        dbo = ROOT / "Tables" / "dbo"
        dbo.mkdir(parents=True, exist_ok=True)
        return str(dbo), None


def _cast_null_type(typ: pa.DataType) -> pa.DataType:
    if pa.types.is_null(typ):
        return pa.string()
    if pa.types.is_struct(typ):
        return pa.struct([
            pa.field(f.name, _cast_null_type(f.type), nullable=f.nullable)
            for f in typ
        ])
    if pa.types.is_list(typ) or pa.types.is_large_list(typ) or pa.types.is_fixed_size_list(typ):
        return pa.list_(_cast_null_type(typ.value_type))
    if pa.types.is_map(typ):
        return pa.map_(_cast_null_type(typ.key_type), _cast_null_type(typ.item_type))
    return typ


def _cast_nulls(table: pa.Table) -> pa.Table:
    new_fields, new_cols = [], []
    for i, field in enumerate(table.schema):
        new_type = _cast_null_type(field.type)
        col = table.column(i)
        if new_type != field.type:
            col = col.cast(new_type)
        new_fields.append(pa.field(field.name, new_type, nullable=field.nullable))
        new_cols.append(col)
    return pa.Table.from_arrays(new_cols, schema=pa.schema(new_fields))


def _arrow(con, sql):
    rel = con.sql(sql)
    if hasattr(rel, "to_arrow_table"):
        return rel.to_arrow_table()
    if hasattr(rel, "fetch_arrow_table"):
        return rel.fetch_arrow_table()
    return rel.arrow()


def _one(con, sql):
    rows = con.sql(sql).fetchall()
    return rows[0] if rows else None


def _write(name, table, base, opts):
    uri = f"{base}/{name}"
    write_deltalake(
        uri,
        _cast_nulls(table),
        mode="overwrite",
        schema_mode="overwrite",
        storage_options=opts,
    )
    return uri


def _exists(name, base, opts) -> bool:
    uri = f"{base}/{name}"
    try:
        DeltaTable(uri, storage_options=opts)
        return True
    except Exception:
        return False


base, opts = _storage()
print(f"dbo path: {base}")

missing = [name for name, _hint in REQUIRED_TABLES.items() if not _exists(name, base, opts)]
if missing:
    hints = "; ".join(f"{n}: {REQUIRED_TABLES[n]}" for n in missing)
    raise SystemExit(f"FAIL missing lakehouse tables: {hints}")

con = duckdb.connect()
for name in REQUIRED_TABLES:
    uri = f"{base}/{name}"
    con.register(name, DeltaTable(uri, storage_options=opts).to_pyarrow_dataset())
    print(f"loaded {name}")

# Date column types must match dim_ga4_date.date (DATE) before Direct Lake relationships.
date_type = str(DeltaTable(f"{base}/dim_ga4_date", storage_options=opts).to_pyarrow_dataset().schema.field("date").type)
date_checks = [
    ("gold_ga4_sessions", "session_date"),
    ("gold_ga4_daily_kpis", "session_date"),
    ("gold_ga4_funnel_daily", "session_date"),
    ("gold_ga4_items_daily", "event_date"),
    ("gold_ga4_landing_pages", "session_date"),
    ("gold_ga4_page_sessions", "event_date"),
]
date_errors = []
for table, col in date_checks:
    typ = str(DeltaTable(f"{base}/{table}", storage_options=opts).to_pyarrow_dataset().schema.field(col).type)
    print(f"{table}.{col} type: {typ}")
    if typ != date_type:
        date_errors.append(f"{table}.{col} is {typ} but dim_ga4_date.date is {date_type}")
if date_errors:
    raise SystemExit("FAIL date types: " + "; ".join(date_errors))

channel = _arrow(con, """
SELECT channel_group
FROM (
    SELECT channel_group FROM gold_ga4_sessions
    UNION
    SELECT channel_group FROM gold_ga4_daily_kpis
    UNION
    SELECT channel_group FROM gold_ga4_funnel_daily
)
WHERE channel_group IS NOT NULL
ORDER BY channel_group
""")
device = _arrow(con, """
SELECT device_category
FROM (
    SELECT device_category FROM gold_ga4_sessions
    UNION
    SELECT device_category FROM gold_ga4_daily_kpis
    UNION
    SELECT device_category FROM gold_ga4_funnel_daily
)
WHERE device_category IS NOT NULL
ORDER BY device_category
""")

chan_uri = _write("dim_ga4_channel", channel, base, opts)
dev_uri = _write("dim_ga4_device", device, base, opts)
print(f"wrote dim_ga4_channel: {chan_uri} rows={len(channel)}")
print(f"wrote dim_ga4_device: {dev_uri} rows={len(device)}")
con.register("dim_ga4_channel", DeltaTable(chan_uri, storage_options=opts).to_pyarrow_dataset())
con.register("dim_ga4_device", DeltaTable(dev_uri, storage_options=opts).to_pyarrow_dataset())

null_counts = _one(con, """
SELECT
    COUNT(*) FILTER (WHERE s.channel_group IS NULL)::BIGINT,
    COUNT(*) FILTER (WHERE s.device_category IS NULL)::BIGINT,
    COUNT(*) FILTER (WHERE s.landing_page_path IS NULL)::BIGINT
FROM gold_ga4_sessions s
""")
print(
    "sessions with null keys: "
    f"channel_group={null_counts[0]:,} device_category={null_counts[1]:,} "
    f"landing_page_path={null_counts[2]:,}"
)

missing_paths = _arrow(con, """
WITH fact_paths AS (
    SELECT DISTINCT page_path FROM gold_ga4_page_sessions WHERE page_path IS NOT NULL
    UNION
    SELECT DISTINCT landing_page_path FROM gold_ga4_landing_pages WHERE landing_page_path IS NOT NULL
    UNION
    SELECT DISTINCT landing_page_path FROM gold_ga4_sessions WHERE landing_page_path IS NOT NULL
)
SELECT f.page_path
FROM fact_paths f
ANTI JOIN dim_ga4_page_type d ON d.page_path = f.page_path
ORDER BY f.page_path
""")
print(f"page paths in facts but not in dim_ga4_page_type: {len(missing_paths):,}")
if len(missing_paths):
    print(missing_paths.slice(0, min(20, len(missing_paths))))
    dim = _arrow(con, f"""
    WITH paths AS (
        SELECT DISTINCT page_path FROM gold_ga4_page_sessions WHERE page_path IS NOT NULL
        UNION
        SELECT DISTINCT landing_page_path AS page_path FROM gold_ga4_landing_pages WHERE landing_page_path IS NOT NULL
        UNION
        SELECT DISTINCT landing_page_path AS page_path FROM gold_ga4_sessions WHERE landing_page_path IS NOT NULL
        UNION
        SELECT DISTINCT page_path FROM dim_ga4_page_type WHERE page_path IS NOT NULL
    ),
    n AS (
        SELECT
            page_path,
            regexp_replace(
                regexp_replace(
                    lower(replace(replace(page_path, '+', ' '), '-', ' ')),
                    '/+',
                    '/',
                    'g'
                ),
                '/+$',
                ''
            ) AS path_norm
        FROM paths
    ),
    segs AS (
        SELECT
            page_path,
            path_norm,
            string_split(trim(path_norm, '/'), '/') AS parts
        FROM n
    ),
    typed AS (
        SELECT
            page_path,
            path_norm,
            CASE
                WHEN len(parts) >= 1 AND parts[len(parts)] IN ('quickview', 'home') AND len(parts) >= 2
                    THEN parts[len(parts) - 1]
                WHEN len(parts) >= 1 THEN parts[len(parts)]
            END AS last_seg
        FROM segs
    )
    SELECT
        page_path,
        {PAGE_TYPE_SQL} AS page_type
    FROM typed
    ORDER BY page_path
    """)
    dim_uri = _write("dim_ga4_page_type", dim, base, opts)
    print(f"rebuilt dim_ga4_page_type: {dim_uri} rows={len(dim)} (was missing {len(missing_paths)} keys)")
    con.register("dim_ga4_page_type", DeltaTable(dim_uri, storage_options=opts).to_pyarrow_dataset())
    leftover = _one(con, """
    WITH fact_paths AS (
        SELECT DISTINCT page_path FROM gold_ga4_page_sessions WHERE page_path IS NOT NULL
        UNION
        SELECT DISTINCT landing_page_path FROM gold_ga4_landing_pages WHERE landing_page_path IS NOT NULL
        UNION
        SELECT DISTINCT landing_page_path FROM gold_ga4_sessions WHERE landing_page_path IS NOT NULL
    )
    SELECT COUNT(*)::BIGINT
    FROM fact_paths f
    ANTI JOIN dim_ga4_page_type d ON d.page_path = f.page_path
    """)[0]
    if leftover:
        raise SystemExit(f"FAIL dim_ga4_page_type still missing {leftover} fact keys after rebuild")
    print("PASS dim_ga4_page_type covers every fact path after rebuild")
else:
    print("kept existing dim_ga4_page_type (every fact path is present)")

dup_chan, n_chan = _one(con, "SELECT COUNT(*) - COUNT(DISTINCT channel_group), COUNT(*) FROM dim_ga4_channel")
dup_dev, n_dev = _one(con, "SELECT COUNT(*) - COUNT(DISTINCT device_category), COUNT(*) FROM dim_ga4_device")
dup_page, n_page = _one(con, "SELECT COUNT(*) - COUNT(DISTINCT page_path), COUNT(*) FROM dim_ga4_page_type")
print(f"dim_ga4_channel rows={n_chan:,} duplicate_keys={dup_chan:,}")
print(f"dim_ga4_device rows={n_dev:,} duplicate_keys={dup_dev:,}")
print(f"dim_ga4_page_type rows={n_page:,} duplicate_keys={dup_page:,}")
print("channel groups:")
con.sql("SELECT channel_group, COUNT(*)::BIGINT AS n FROM gold_ga4_sessions GROUP BY 1 ORDER BY n DESC").show()
print("device categories:")
con.sql("SELECT device_category, COUNT(*)::BIGINT AS n FROM gold_ga4_sessions GROUP BY 1 ORDER BY n DESC").show()
if dup_chan or dup_dev or dup_page:
    raise SystemExit("FAIL conformed dims have duplicate keys")
print("PASS conformed dimensions")


In [ ]:
# Inspect lakehouse column types (expect Date for session_date / event_date / date)
def _print_gold_types() -> None:
    try:
        from sempy_labs.lakehouse import get_lakehouse_columns

        cols = get_lakehouse_columns(lakehouse=LAKEHOUSE, workspace=WORKSPACE)
        print("get_lakehouse_columns rows:", len(cols))
        names = {c.lower(): c for c in cols.columns}
        type_col = next((names[k] for k in ("type", "data type", "data_type") if k in names), None)
        table_col = next((names[k] for k in ("table name", "table", "name") if k in names), None)
        col_col = next((names[k] for k in ("column name", "column") if k in names), None)
        if type_col and table_col and col_col:
            mask = cols[col_col].isin(
                [
                    "date",
                    "session_date",
                    "event_date",
                    "channel_group",
                    "device_category",
                    "page_path",
                    "landing_page_path",
                ]
            )
            print(cols.loc[mask, [table_col, col_col, type_col]].to_string(index=False))
    except Exception as ex:
        print("get_lakehouse_columns skipped:", type(ex).__name__, ex)


_print_gold_types()


## 1 · Create the model only if it is absent

`generate_direct_lake_semantic_model` is called with `overwrite=False` and `use_sql_endpoint=False` (Direct Lake on OneLake, not the SQL endpoint). If **GA4 Ecommerce** already exists, this cell skips generation and the next cells update it via TOM. The model is never deleted or overwritten.


In [ ]:
items = fabric.list_items(workspace=WORKSPACE)
# Column names vary slightly by sempy version
name_col = next(c for c in items.columns if c.lower() in {"display name", "name"})
type_col = next(c for c in items.columns if c.lower() == "type")
models = items[items[type_col].astype(str).str.lower().isin({"semanticmodel", "semantic model"})]
names = set(models[name_col].astype(str))
print("semantic models:", sorted(names) or "(none)")

created = False
if DATASET in names:
    print(f"Found existing '{DATASET}' — will update relationships/measures in place (no overwrite).")
else:
    extra = [n for n in names if n.lower() == DATASET.lower() and n != DATASET]
    if extra:
        raise SystemExit(
            f"A similarly named model exists {extra} but is not exactly '{DATASET}'. "
            "Refusing to create or overwrite. Rename it or set DATASET to match."
        )
    print(f"Creating Direct Lake model '{DATASET}' on {LAKEHOUSE} …")
    generate_direct_lake_semantic_model(
        dataset=DATASET,
        tables=TABLES,
        source=LAKEHOUSE,
        source_type="Lakehouse",
        source_workspace=WORKSPACE,
        use_sql_endpoint=False,
        workspace=WORKSPACE,
        refresh=True,
        inherit_descriptions=True,
        overwrite=False,
    )
    created = True
    print("generated + framed")

print("created_this_run:", created)


## 2 · TOM: relationships, date table, formats, measures

Requires XMLA read/write. Changes save when the `with connect_semantic_model(..., readonly=False)` block exits.


In [ ]:
TABLE_ALIASES = {
    "Date": ["Date", "dim_ga4_date", "dbo.dim_ga4_date"],
    "Channel": ["Channel", "dim_ga4_channel", "dbo.dim_ga4_channel"],
    "Device": ["Device", "dim_ga4_device", "dbo.dim_ga4_device"],
    "Page Type": ["Page Type", "dim_ga4_page_type", "dbo.dim_ga4_page_type"],
    "Sessions": ["Sessions", "gold_ga4_sessions", "dbo.gold_ga4_sessions"],
    "Daily KPIs": ["Daily KPIs", "gold_ga4_daily_kpis", "dbo.gold_ga4_daily_kpis"],
    "Funnel Daily": ["Funnel Daily", "gold_ga4_funnel_daily", "dbo.gold_ga4_funnel_daily"],
    "Items Daily": ["Items Daily", "gold_ga4_items_daily", "dbo.gold_ga4_items_daily"],
    "Landing Pages": ["Landing Pages", "gold_ga4_landing_pages", "dbo.gold_ga4_landing_pages"],
    "Page Sessions": ["Page Sessions", "gold_ga4_page_sessions", "dbo.gold_ga4_page_sessions"],
}

P01 = r"Pages\01 Overview"
P02 = r"Pages\02 Acquisition"
P03 = r"Pages\03 Content"
P04 = r"Pages\04 Funnel & Products"
OV = f"{P01};{P02}"
OV_CONTENT = f"{P01};{P02};{P03}"
CONTENT = f"{P01};{P03}"
FUNNEL = f"{P01};{P04}"

MEASURES = [
    {
        "name": "Sessions",
        "folder": OV_CONTENT,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Count of sessions from gold_ga4_sessions (one row per session). Do not SUM the pre-aggregated sessions columns on Daily KPIs or Landing Pages.",
        "expr": "COUNTROWS('Sessions')",
    },
    {
        "name": "Users",
        "folder": OV_CONTENT,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Distinct user_pseudo_id from gold_ga4_sessions. Users do not add up across channel, device, or page breakdowns.",
        "expr": "DISTINCTCOUNT('Sessions'[user_pseudo_id])",
    },
    {
        "name": "New Users",
        "folder": OV,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Distinct users whose first session (is_new_user) is in the current filter context.",
        "expr": "CALCULATE(\n    DISTINCTCOUNT('Sessions'[user_pseudo_id]),\n    'Sessions'[is_new_user] = TRUE()\n)",
    },
    {
        "name": "Engaged Sessions",
        "folder": OV_CONTENT,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Sessions flagged engaged (GA4 session_engaged = 1) in gold_ga4_sessions.",
        "expr": "CALCULATE(\n    [Sessions],\n    'Sessions'[engaged] = TRUE()\n)",
    },
    {
        "name": "Engagement Rate",
        "folder": OV_CONTENT,
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Engaged Sessions divided by Sessions. This is the GA4 engagement rate, not 1 minus Universal Analytics bounce rate. Computed with DIVIDE; do not average the engagement_rate column.",
        "expr": "DIVIDE([Engaged Sessions], [Sessions])",
    },
    {
        "name": "Purchases",
        "folder": f"{P01};{P02};{P04}",
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Count of purchase events in gold_ga4_sessions (SUM of purchases). A session can have more than one purchase.",
        "expr": "SUM('Sessions'[purchases])",
    },
    {
        "name": "Revenue",
        "folder": f"{P01};{P02};{P03};{P04}",
        "format": "$#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Sum of purchase revenue in USD from gold_ga4_sessions.",
        "expr": "SUM('Sessions'[revenue])",
    },
    {
        "name": "Conversion Rate",
        "folder": f"{P01};{P02};{P03};{P04}",
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Share of sessions with at least one purchase. Computed with DIVIDE; do not average the conversion_rate column.",
        "expr": "DIVIDE(\n    CALCULATE([Sessions], 'Sessions'[purchases] > 0),\n    [Sessions]\n)",
    },
    {
        "name": "AOV",
        "folder": OV,
        "format": "$#,##0.00",
        "table": "_Measures",
        "hidden": False,
        "description": "Average order value: Revenue divided by Purchases (purchase events). Computed with DIVIDE; do not average the aov column.",
        "expr": "DIVIDE([Revenue], [Purchases])",
    },
    {
        "name": "Revenue per Session",
        "folder": OV_CONTENT,
        "format": "$#,##0.00",
        "table": "_Measures",
        "hidden": False,
        "description": "Revenue divided by Sessions. Computed with DIVIDE; do not average the revenue_per_session column.",
        "expr": "DIVIDE([Revenue], [Sessions])",
    },
    {
        "name": "Revenue Share of Top Channel",
        "folder": OV,
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Share of Revenue that comes from the single largest channel_group in the current selection (ALLSELECTED on Channel). A concentration-risk KPI.",
        "expr": "VAR TotalRevenue =\n    CALCULATE([Revenue], ALLSELECTED('Channel'[channel_group]))\nVAR TopChannelRevenue =\n    MAXX(\n        ALLSELECTED('Channel'[channel_group]),\n        [Revenue]\n    )\nRETURN\nDIVIDE(TopChannelRevenue, TotalRevenue)",
    },
    {
        "name": "Pageviews",
        "folder": CONTENT,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Sum of pageviews from gold_ga4_page_sessions (page_view events with a page_location).",
        "expr": "SUM('Page Sessions'[pageviews])",
    },
    {
        "name": "View Item Sessions",
        "folder": P04,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Sessions that fired view_item, summed from gold_ga4_funnel_daily (additive at date × channel × device).",
        "expr": "SUM('Funnel Daily'[view_item_sessions])",
    },
    {
        "name": "Add to Cart Sessions",
        "folder": P04,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Sessions that fired add_to_cart, summed from gold_ga4_funnel_daily.",
        "expr": "SUM('Funnel Daily'[add_to_cart_sessions])",
    },
    {
        "name": "Begin Checkout Sessions",
        "folder": P04,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Sessions that fired begin_checkout, summed from gold_ga4_funnel_daily.",
        "expr": "SUM('Funnel Daily'[begin_checkout_sessions])",
    },
    {
        "name": "Add Payment Info Sessions",
        "folder": P04,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Sessions that fired add_payment_info, summed from gold_ga4_funnel_daily.",
        "expr": "SUM('Funnel Daily'[add_payment_info_sessions])",
    },
    {
        "name": "Purchase Sessions",
        "folder": P04,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Sessions that fired purchase, summed from gold_ga4_funnel_daily. Distinct from Purchases, which counts purchase events.",
        "expr": "SUM('Funnel Daily'[purchase_sessions])",
    },
    {
        "name": "View Item to Cart Rate",
        "folder": P04,
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Add to Cart Sessions divided by View Item Sessions.",
        "expr": "DIVIDE([Add to Cart Sessions], [View Item Sessions])",
    },
    {
        "name": "Cart to Checkout Rate",
        "folder": P04,
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Begin Checkout Sessions divided by Add to Cart Sessions.",
        "expr": "DIVIDE([Begin Checkout Sessions], [Add to Cart Sessions])",
    },
    {
        "name": "Checkout to Payment Rate",
        "folder": P04,
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Add Payment Info Sessions divided by Begin Checkout Sessions.",
        "expr": "DIVIDE([Add Payment Info Sessions], [Begin Checkout Sessions])",
    },
    {
        "name": "Payment to Purchase Rate",
        "folder": P04,
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Purchase Sessions divided by Add Payment Info Sessions.",
        "expr": "DIVIDE([Purchase Sessions], [Add Payment Info Sessions])",
    },
    {
        "name": "View Item to Purchase Rate",
        "folder": P04,
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Purchase Sessions divided by View Item Sessions (overall funnel conversion).",
        "expr": "DIVIDE([Purchase Sessions], [View Item Sessions])",
    },
    {
        "name": "Item Views",
        "folder": P04,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Item-level view_item counts from gold_ga4_items_daily (related-item noise already removed in gold).",
        "expr": "SUM('Items Daily'[item_views])",
    },
    {
        "name": "Add to Carts",
        "folder": P04,
        "format": "#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Item-level add_to_cart counts from gold_ga4_items_daily.",
        "expr": "SUM('Items Daily'[add_to_carts])",
    },
    {
        "name": "Item Revenue",
        "folder": P04,
        "format": "$#,##0",
        "table": "_Measures",
        "hidden": False,
        "description": "Item-level purchase revenue from gold_ga4_items_daily. May differ slightly from Revenue, which is session purchase_revenue_in_usd.",
        "expr": "SUM('Items Daily'[item_revenue])",
    },
    {
        "name": "Add-to-Cart Rate",
        "folder": P04,
        "format": "0.00%",
        "table": "_Measures",
        "hidden": False,
        "description": "Add to Carts divided by Item Views. Computed with DIVIDE.",
        "expr": "DIVIDE([Add to Carts], [Item Views])",
    },
]


def resolve_name(tom, logical: str) -> str:
    have = [t.Name for t in tom.model.Tables]
    for alias in TABLE_ALIASES[logical]:
        if alias in have:
            return alias
    raise KeyError(f"Table '{logical}' not in model. Have: {have}")


def col_dtype(tom, table: str, column: str) -> str:
    return str(tom.model.Tables[table].Columns[column].DataType)


def upsert_relationship(tom, from_table, from_col, to_table, to_col):
    for rel in list(tom.model.Relationships):
        if (
            rel.FromTable.Name == from_table
            and rel.FromColumn.Name == from_col
            and rel.ToTable.Name == to_table
            and rel.ToColumn.Name == to_col
        ):
            tom.remove_object(rel)
    tom.add_relationship(
        from_table=from_table,
        from_column=from_col,
        to_table=to_table,
        to_column=to_col,
        from_cardinality="Many",
        to_cardinality="One",
        cross_filtering_behavior="OneDirection",
        is_active=True,
    )


def upsert_measure(tom, host: str, m: dict) -> None:
    current = next((meas for meas in tom.all_measures() if meas.Name == m["name"]), None)
    if current is not None and current.Parent.Name != host:
        tom.remove_object(current)
        current = None
    if current is not None:
        tom.update_measure(
            measure_name=m["name"],
            expression=m["expr"],
            format_string=m["format"],
            hidden=m["hidden"],
            description=m["description"],
            display_folder=m["folder"],
        )
    else:
        tom.add_measure(
            table_name=host,
            measure_name=m["name"],
            expression=m["expr"],
            format_string=m["format"],
            hidden=m["hidden"],
            description=m["description"],
            display_folder=m["folder"],
        )


with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    have = {t.Name for t in tom.model.Tables}
    missing = {}
    for logical, entity in TABLES.items():
        if not any(alias in have for alias in TABLE_ALIASES[logical]):
            missing[logical] = entity
    if missing:
        print("adding missing Direct Lake tables:", missing)
        tom.add_direct_lake_tables(
            tables=missing,
            source=LAKEHOUSE,
            source_type="Lakehouse",
            source_workspace=WORKSPACE,
            use_sql_endpoint=False,
        )

    resolved = {logical: resolve_name(tom, logical) for logical in TABLES}
    print("resolved tables:", resolved)

    date_tbl = resolved["Date"]
    date_type = col_dtype(tom, date_tbl, "date")
    print(f"Date[date] type: {date_type}")
    fact_dates = [
        ("Sessions", "session_date"),
        ("Daily KPIs", "session_date"),
        ("Funnel Daily", "session_date"),
        ("Items Daily", "event_date"),
        ("Landing Pages", "session_date"),
        ("Page Sessions", "event_date"),
    ]
    for fact, col in fact_dates:
        ft = resolved[fact]
        typ = col_dtype(tom, ft, col)
        print(f"{ft}[{col}] type: {typ}")
        if typ != date_type:
            raise SystemExit(
                f"{ft}[{col}] is {typ} but Date[date] is {date_type}. "
                "Date columns must be Date in Delta. Rerun the gold notebooks, then this notebook."
            )

    for from_logical, from_col, to_logical, to_col in RELATIONSHIPS:
        upsert_relationship(tom, resolved[from_logical], from_col, resolved[to_logical], to_col)
        print(f"rel {from_logical}[{from_col}] → {to_logical}[{to_col}]")

    tom.mark_as_date_table(table_name=resolved["Date"], column_name="date", validate=False)

    d = resolved["Date"]
    sort_pairs = [
        (d, "month_name", "month_num"),
        (d, "month_short", "month_num"),
        (d, "day_name", "day_of_week_num"),
        (d, "day_short", "day_of_week_num"),
    ]
    for tbl, col, by in sort_pairs:
        have_cols = [c.Name for c in tom.model.Tables[tbl].Columns]
        if col in have_cols and by in have_cols:
            tom.set_sort_by_column(tbl, col, by)

    no_sum = {
        d: [
            "date_key",
            "year",
            "quarter",
            "month_num",
            "iso_week",
            "day_of_month",
            "day_of_week_num",
            "days_from_start",
        ],
        resolved["Sessions"]: [
            "ga_session_id",
            "pageviews",
            "engagement_time_sec",
            "purchases",
            "revenue",
            "transactions",
        ],
        resolved["Daily KPIs"]: [
            "sessions",
            "users",
            "new_users",
            "engaged_sessions",
            "engagement_rate",
            "purchases",
            "revenue",
            "conversion_rate",
            "aov",
            "revenue_per_session",
        ],
        resolved["Funnel Daily"]: [
            "view_item_sessions",
            "add_to_cart_sessions",
            "begin_checkout_sessions",
            "add_payment_info_sessions",
            "purchase_sessions",
        ],
        resolved["Items Daily"]: [
            "item_id",
            "item_views",
            "add_to_carts",
            "checkouts",
            "purchases",
            "item_revenue",
        ],
        resolved["Landing Pages"]: [
            "sessions",
            "users",
            "engaged_sessions",
            "purchases",
            "revenue",
            "conversion_rate",
            "revenue_per_session",
        ],
        resolved["Page Sessions"]: [
            "pageviews",
            "sessions",
            "users",
            "entrances",
            "engaged_sessions",
            "avg_engagement_time_sec",
            "exits",
        ],
        resolved["Channel"]: ["channel_group"],
        resolved["Device"]: ["device_category"],
        resolved["Page Type"]: ["page_path"],
    }
    for tbl, cols in no_sum.items():
        have_cols = {c.Name for c in tom.model.Tables[tbl].Columns}
        for c in cols:
            if c in have_cols:
                tom.set_summarize_by(tbl, c, "None")

    hide = {
        resolved["Sessions"]: [
            "session_id",
            "user_pseudo_id",
            "ga_session_id",
            "session_date",
            "channel_group",
            "device_category",
            "landing_page_path",
            "pageviews",
            "engagement_time_sec",
            "purchases",
            "revenue",
            "transactions",
        ],
        resolved["Daily KPIs"]: [
            "session_date",
            "channel_group",
            "device_category",
            "sessions",
            "users",
            "new_users",
            "engaged_sessions",
            "engagement_rate",
            "purchases",
            "revenue",
            "conversion_rate",
            "aov",
            "revenue_per_session",
        ],
        resolved["Funnel Daily"]: [
            "session_date",
            "channel_group",
            "device_category",
            "view_item_sessions",
            "add_to_cart_sessions",
            "begin_checkout_sessions",
            "add_payment_info_sessions",
            "purchase_sessions",
        ],
        resolved["Items Daily"]: [
            "event_date",
            "item_views",
            "add_to_carts",
            "checkouts",
            "purchases",
            "item_revenue",
        ],
        resolved["Landing Pages"]: [
            "session_date",
            "landing_page_path",
            "sessions",
            "users",
            "engaged_sessions",
            "purchases",
            "revenue",
            "conversion_rate",
            "revenue_per_session",
        ],
        resolved["Page Sessions"]: [
            "event_date",
            "page_path",
            "pageviews",
            "sessions",
            "users",
            "entrances",
            "engaged_sessions",
            "avg_engagement_time_sec",
            "exits",
        ],
        resolved["Date"]: ["date_key", "days_from_start"],
    }
    mdx_off = {
        "session_id",
        "user_pseudo_id",
        "ga_session_id",
        "date_key",
        "days_from_start",
    }
    for tbl, cols in hide.items():
        have_cols = {c.Name for c in tom.model.Tables[tbl].Columns}
        for c in cols:
            if c in have_cols:
                tom.model.Tables[tbl].Columns[c].IsHidden = True
                if c in mdx_off:
                    tom.set_is_available_in_mdx(tbl, c, False)

    def ensure_hierarchy(table, name, columns):
        have_h = {h.Name for h in tom.model.Tables[table].Hierarchies}
        have_c = {c.Name for c in tom.model.Tables[table].Columns}
        if name in have_h:
            return
        if not all(c in have_c for c in columns):
            print("skip hierarchy", name, "missing columns")
            return
        tom.add_hierarchy(table_name=table, hierarchy_name=name, columns=columns)

    ensure_hierarchy(d, "Calendar", ["year", "month_name", "date"])

    host = MEASURES[0]["table"] if MEASURES else "_Measures"
    if host != "_Measures":
        raise SystemExit(f"base measures must target _Measures, not {host}")

    def ensure_measures_table(tom, name: str = "_Measures") -> None:
        have = [t.Name for t in tom.model.Tables]
        if name not in have:
            tom.add_calculated_table(
                name=name,
                expression='ROW("Dummy", BLANK())',
                description="Measures home. The Dummy column is hidden so the table shows the measure icon.",
                hidden=False,
            )
            print("created calculated table", name)
        elif not tom.is_calculated_table(name):
            raise SystemExit(f"{name} exists but is not a calculated table")
        for col in tom.model.Tables[name].Columns:
            col.IsHidden = True

    ensure_measures_table(tom, host)

    tom.model.DiscourageImplicitMeasures = True

    for m in MEASURES:
        upsert_measure(tom, host, m)
        print("measure:", m["name"], "on", host, "folder:", m["folder"])

    print("TOM updates queued; saved on context exit")


## 3 · Frame (refresh) and validate gold totals


In [ ]:
refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="full")
print("refresh/framing complete")

with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=True) as tom:
    print("is_direct_lake:", tom.is_direct_lake())
    print(
        "date table marked:",
        tom.is_date_table(
            next(t.Name for t in tom.all_date_tables()) if tom.has_date_table() else "?"
        ),
    )
    print("relationships:")
    for rel in tom.model.Relationships:
        print(
            f"  {rel.FromTable.Name}[{rel.FromColumn.Name}] → "
            f"{rel.ToTable.Name}[{rel.ToColumn.Name}] "
            f"active={rel.IsActive} xfilter={rel.CrossFilteringBehavior}"
        )
    print("measures:")
    for m in tom.all_measures():
        print(f"  {m.Parent.Name}[{m.Name}] folder={m.DisplayFolder!r}")

dax = """
EVALUATE
ROW(
    "Sessions", [Sessions],
    "Revenue", [Revenue],
    "Pageviews", [Pageviews]
)
"""
result = fabric.evaluate_dax(dataset=DATASET, workspace=WORKSPACE, dax_string=dax)
print(result)

row = result.iloc[0]
failed = []
for label, expected in EXPECTED.items():
    col = next((c for c in result.columns if label in str(c)), None)
    got = None if col is None else row[col]
    ok = col is not None and got is not None and abs(float(got) - float(expected)) <= 1
    status = "PASS" if ok else "FAIL"
    print(f"  {status} {label}: got={got} expected={expected:,.0f}")
    if not ok:
        failed.append(label)

print("OVERALL:", "FAIL" if failed else "PASS")
if failed:
    raise AssertionError("gold total check failed: " + ", ".join(failed))
